# Orthanc Study Metadata Test
 
Use this notebook to test custom study-level metadata fields on Orthanc: 
- `groundTruthReport` 
- `mammoReport` 
- `medGemmaReport` 
- `false_report_finding`
- `missing_finding`
- `mischaracterization_finding`
- `misidentification_finding`
- `incorrect_birads_assessment_finding`
- `breast_denstity_mismatch`

## 1) Configuration 
 
Set these env vars before launching Jupyter if needed: 
- `ORTHANC_BASE_URL` (default: `http://localhost:8042/pacs`) 
- `ORTHANC_USERNAME` (optional) 
- `ORTHANC_PASSWORD` (optional) 
 
If your server does not use the `/pacs` prefix, set `ORTHANC_BASE_URL` accordingly (for example `http://localhost:8042`).

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
import json
import base64
import time
import ssl
import socket
import urllib.parse
import urllib.request
import urllib.error
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ORTHANC_BASE_URL = os.getenv('ORTHANC_BASE_URL')
ORTHANC_USERNAME = os.getenv('ORTHANC_USERNAME')
ORTHANC_PASSWORD = os.getenv('ORTHANC_PASSWORD')
ORTHANC_VERIFY = os.getenv('ORTHANC_VERIFY', 'true').lower() not in ('false', '0', 'no')

print('ORTHANC_BASE_URL =', ORTHANC_BASE_URL)
print('Auth configured =', bool(ORTHANC_USERNAME and ORTHANC_PASSWORD))

In [ ]:
import base64, urllib.request

auth = base64.b64encode(f"{ORTHANC_USERNAME}:{ORTHANC_PASSWORD}".encode()).decode()
req = urllib.request.Request(f"{ORTHANC_BASE_URL}/system")
req.add_header("Authorization", f"Basic {auth}")

with urllib.request.urlopen(req, timeout=10) as r:
    print(r.status, r.read()[:120])

In [ ]:
REQUEST_TIMEOUT = 30
REQUEST_RETRIES = 3
REQUEST_BACKOFF = 2.0
DELETE_PAUSE_SECONDS = 0.2


def _auth_header():
    if not (ORTHANC_USERNAME and ORTHANC_PASSWORD):
        return None
    token = base64.b64encode(f"{ORTHANC_USERNAME}:{ORTHANC_PASSWORD}".encode('utf-8')).decode('utf-8')
    return f"Basic {token}"


_session = None

def _get_session():
    global _session
    if _session is None:
        session = requests.Session()
        retries = Retry(
            total=REQUEST_RETRIES,
            connect=REQUEST_RETRIES,
            read=REQUEST_RETRIES,
            backoff_factor=REQUEST_BACKOFF,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=frozenset(['GET', 'PUT', 'DELETE', 'POST'])
        )
        adapter = HTTPAdapter(max_retries=retries)
        session.mount('http://', adapter)
        session.mount('https://', adapter)
        _session = session
    return _session


def _request(method, path, *, params=None, body=None, content_type=None, timeout=REQUEST_TIMEOUT):
    query = f"?{urllib.parse.urlencode(params)}" if params else ''
    url = f"{ORTHANC_BASE_URL}{path}{query}"
    headers = {}
    auth = _auth_header()
    if auth:
        headers['Authorization'] = auth
    if content_type:
        headers['Content-Type'] = content_type

    data = None
    if body is not None:
        data = body if isinstance(body, (bytes, bytearray)) else str(body).encode('utf-8')

    session = _get_session()
    try:
        resp = session.request(
            method,
            url,
            headers=headers,
            data=data,
            timeout=timeout,
            verify=ORTHANC_VERIFY,
        )
        resp.raise_for_status()
        return {
            'status': resp.status_code,
            'headers': dict(resp.headers),
            'bytes': resp.content,
            'text': resp.text
        }
    except requests.exceptions.RequestException as e:
        print(f"Request exception for {method} {url}: {e}")
        raise


def orthanc_get(path, params=None):
    return _request('GET', path, params=params)


def orthanc_put_text(path, value):
    return _request('PUT', path, body=value, content_type='text/plain')


def orthanc_delete(path):
    return _request('DELETE', path)


## 2) Find Orthanc Study ID from StudyInstanceUID

In [ ]:
def get_studies_with_uid():
    params = {'expand': 1, 'requestedTags': 'StudyInstanceUID'}
    r = orthanc_get('/studies', params=params)
    return json.loads(r['text'])

def get_orthanc_study_id(study_instance_uid):
    studies = get_studies_with_uid()
    for s in studies:
        uid = (s.get('RequestedTags') or {}).get('StudyInstanceUID')
        if uid == study_instance_uid:
            return s.get('ID')
    return None

In [ ]:
# Set your StudyInstanceUID here
# study_instance_uid = 'PUT_YOUR_STUDY_INSTANCE_UID_HERE'
study_instance_uid = '1.2.840.113654.2.70.1.213012051129742288091890829040441839772'

orthanc_study_id = get_orthanc_study_id(study_instance_uid)
print('StudyInstanceUID:', study_instance_uid)
print('Orthanc study ID:', orthanc_study_id)
if not orthanc_study_id:
    raise ValueError('Study not found. Check StudyInstanceUID and ORTHANC_BASE_URL.')

### Or use Patient ID to find Orthanc Study ID

In [ ]:
def get_patients_with_patient_id():
    params = {'expand': 1, 'requestedTags': 'PatientID'}
    r = orthanc_get('/patients', params=params)
    return json.loads(r['text'])

def get_orthanc_study_ids_from_patient_id(patient_id):
    pid = str(patient_id)
    patients = get_patients_with_patient_id()

    # Multiple Orthanc patient records can share the same PatientID
    matching_patients = [
        p for p in patients
        if ((p.get('RequestedTags') or {}).get('PatientID') == pid)
    ]

    study_ids = []
    for p in matching_patients:
        patient_obj = json.loads(orthanc_get(f"/patients/{p['ID']}")['text'])
        study_ids.extend(patient_obj.get('Studies', []))

    # unique, preserve order
    return list(dict.fromkeys(study_ids))

In [ ]:
import json

# 1) load your grouped patient IDs
with open('patient_id_groups.json', 'r') as f:
    patient_groups = json.load(f)

# 2) pick a group from your file: "asymetry", "calcification", "mass", "no-findings"
group_name = 'mass'
patient_ids = patient_groups[group_name]

# 3) example: resolve Orthanc study IDs for first 3 patients in that group
for patient_id in patient_ids[:3]:
    study_ids = get_orthanc_study_ids_from_patient_id(patient_id)
    print(f'PatientID {patient_id} -> StudyIDs: {study_ids}')

In [ ]:
get_orthanc_study_ids_from_patient_id(936948411)

## 2.5) Read in Spreadsheet

In [ ]:
import pandas as pd

df = pd.read_csv('/Users/katelynmorrison/Downloads/upmc_joined_zs_mammo_clip_medgemma.csv')
df_llm_judge_report = pd.read_csv('/Users/katelynmorrison/Downloads/Complete_gpt-4o-mini.csv')

data = {}

# Loop rows
for _, row in df.iterrows():
    patient_id_csv = row['patient_id']
    truth = row['ground_truth_report']
    gemma = row['final_generated_report_zs_medgemma']
    data[patient_id_csv] = {
        'ground_truth_report': truth,
        'final_generated_report_zs_medgemma': gemma
    }

for _, row in df_llm_judge_report.iterrows():
    patient_id_csv = row['patient_id']
    mammo = row['final_generated_report_zs']
    false_report_finding = row['sig_a_false_report']
    missing_finding = row['sig_b_missing_finding']
    mischaracterization_finding = row['sig_c_mischaracterization']
    misidentification_finding = row['sig_d_location_laterality']
    incorrect_birads_assessment_finding = row['sig_e_incorrect_birads']
    breast_denstity_mismatch = row['sig_f_density_mismatch']
    sig_a_false_report_expl = row['sig_a_false_report_expl']
    sig_b_missing_finding_expl = row['sig_b_missing_finding_expl']
    sig_c_mischaracterization_expl = row['sig_c_mischaracterization_expl']
    sig_d_location_laterality_expl = row['sig_d_location_laterality_expl']
    sig_e_incorrect_birads_expl = row['sig_e_incorrect_birads_expl']
    sig_f_density_mismatch_expl = row['sig_f_density_mismatch_expl']

    if patient_id_csv not in data:
        data[patient_id_csv] = {}

    data[patient_id_csv].update({
        'final_generated_report_zs': mammo,
        'false_report_finding': false_report_finding,
        'missing_finding': missing_finding,
        'mischaracterization_finding': mischaracterization_finding,
        'misidentification_finding': misidentification_finding,
        'incorrect_birads_assessment_finding': incorrect_birads_assessment_finding,
        'breast_denstity_mismatch': breast_denstity_mismatch,
        'false_report_finding_explanation': sig_a_false_report_expl,
        'missing_finding_explanation': sig_b_missing_finding_expl,
        'mischaracterization_finding_explanation': sig_c_mischaracterization_expl,
        'misidentification_finding_explanation': sig_d_location_laterality_expl,
        'incorrect_birads_assessment_finding_explanation': sig_e_incorrect_birads_expl,
        'breast_denstity_mismatch_explanation': sig_f_density_mismatch_expl
    })

In [ ]:
import json

# Choose exactly which patients are allowed
with open('patient_id_groups.json', 'r') as f:
    patient_groups = json.load(f)

group_name = 'calc_pos'  # change to: asymetry, calcification, mass, no-findings
target_patient_ids = [str(x) for x in patient_groups[group_name]]

METADATA_KEYS = [
    'groundTruthReport',
    'mammoReport',
    'medGemmaReport',
    'false_report_finding',
    'missing_finding',
    'mischaracterization_finding',
    'misidentification_finding',
    'incorrect_birads_assessment_finding',
    'breast_denstity_mismatch',
    'false_report_finding_explanation',
    'missing_finding_explanation',
    'mischaracterization_finding_explanation',
    'misidentification_finding_explanation',
    'incorrect_birads_assessment_finding_explanation',
    'breast_denstity_mismatch_explanation',
]

def row_for_patient(patient_id_str):
    # your `data` dict may have int keys or str keys depending on CSV load
    return data.get(int(patient_id_str)) or data.get(patient_id_str)

for patient_id in target_patient_ids:
    row = row_for_patient(patient_id)
    if not row:
        print(f"SKIP patient {patient_id}: not found in data dict")
        continue

    study_ids = get_orthanc_study_ids_from_patient_id(patient_id)
    print(f"PatientID {patient_id} -> StudyIDs: {study_ids}")

    if not study_ids:
        print(f"SKIP patient {patient_id}: no Orthanc studies found")
        continue

    metadata_payload = {
        'groundTruthReport': row['ground_truth_report'],
        'mammoReport': row['final_generated_report_zs'],
        'medGemmaReport': row['final_generated_report_zs_medgemma'],
        'false_report_finding': row['false_report_finding'],
        'missing_finding': row['missing_finding'],
        'mischaracterization_finding': row['mischaracterization_finding'],
        'misidentification_finding': row['misidentification_finding'],
        'incorrect_birads_assessment_finding': row['incorrect_birads_assessment_finding'],
        'breast_denstity_mismatch': row['breast_denstity_mismatch'],
        'false_report_finding_explanation': row['false_report_finding_explanation'],
        'missing_finding_explanation': row['missing_finding_explanation'],
        'mischaracterization_finding_explanation': row['mischaracterization_finding_explanation'],
        'misidentification_finding_explanation': row['misidentification_finding_explanation'],
        'incorrect_birads_assessment_finding_explanation': row['incorrect_birads_assessment_finding_explanation'],
        'breast_denstity_mismatch_explanation': row['breast_denstity_mismatch_explanation'],
    }

    for study_id in study_ids:
        for key in METADATA_KEYS:
            orthanc_put_text(f'/studies/{study_id}/metadata/{key}', metadata_payload[key])
        print(f"WROTE {len(METADATA_KEYS)} metadata keys to study {study_id} (PatientID {patient_id})")

## 4) Read Back Metadata

In [ ]:
for key in ['groundTruthReport', 'mammoReport', 'medGemmaReport']:
    try:
        r = orthanc_get(f'/studies/{orthanc_study_id}/metadata/{key}')
        print(key, r['text'])
    except Exception as e:
        print(key, '(missing or error)', e)


## 5) Optional Cleanup (Delete Metadata Keys) 
 
Uncomment and run if you want to remove test metadata values.

In [ ]:
# Example
# for key in [1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038]:  # replace with actual keys you want to delete
#     orthanc_delete(f'/studies/{orthanc_study_id}/metadata/{key}')
#     print(f'Deleted metadata key: {key}')

## 3.5) Delete Studies Not In Allowed Patient List
Use this cell to find and delete studies that do not belong to the patient IDs in your selected group. Run with `dry_run=True` first to verify the list.

In [ ]:
import json

# Load all allowed patient IDs from every group in the file
with open('patient_id_groups.json', 'r') as f:
    patient_groups = json.load(f)

allowed_patient_ids = [str(x) for group in patient_groups.values() for x in group]
allowed_patient_ids = list(dict.fromkeys(allowed_patient_ids))  # preserve order and dedupe


def get_all_studies():
    r = orthanc_get('/studies', params={'expand': 1})
    return json.loads(r['text'])


def get_all_patients():
    r = orthanc_get('/patients', params={'expand': 1, 'requestedTags': 'PatientID'})
    return json.loads(r['text'])


def get_allowed_study_ids(patient_ids):
    allowed = set(str(x) for x in patient_ids)
    keep = set()
    for patient in get_all_patients():
        pid = (patient.get('RequestedTags') or {}).get('PatientID')
        if pid in allowed:
            patient_obj = json.loads(orthanc_get(f"/patients/{patient['ID']}")['text'])
            keep.update(patient_obj.get('Studies', []))
    return keep


def delete_studies_not_in_allowed_patient_ids(patient_ids, dry_run=True):
    allowed_study_ids = get_allowed_study_ids(patient_ids)
    all_studies = get_all_studies()

    all_study_ids = [study['ID'] for study in all_studies]
    to_delete = [sid for sid in all_study_ids if sid not in allowed_study_ids]

    print('Allowed Patient IDs:', len(set(str(x) for x in patient_ids)))
    print('Total studies found in Orthanc:', len(all_study_ids))
    print('Study IDs to delete:', len(to_delete))
    print('Allowed study IDs:', len(allowed_study_ids))

    if dry_run:
        print('Dry run: no deletes performed. Remove dry_run=True to execute deletes.')
        return to_delete

    deleted = []
    failed = []
    for study_id in to_delete:
        print('Deleting study:', study_id)
        try:
            resp = orthanc_delete(f'/studies/{study_id}')
            if resp['status'] == 200:
                deleted.append(study_id)
            else:
                print('Failed to delete study', study_id, resp['status'], resp.get('text'))
                failed.append(study_id)
        except Exception as e:
            print('Delete failed for', study_id, 'with error:', type(e).__name__, e)
            failed.append(study_id)
        time.sleep(DELETE_PAUSE_SECONDS)

    if failed:
        print('Failed to delete', len(failed), 'studies. See output above.')
    return deleted


def delete_studies_for_patient_id(patient_id, dry_run=True):
    patient_id = str(patient_id)
    study_ids = get_orthanc_study_ids_from_patient_id(patient_id)
    if not study_ids:
        print(f'No Orthanc studies found for PatientID {patient_id}')
        return []

    print(f'Found {len(study_ids)} study(ies) for PatientID {patient_id}: {study_ids}')
    if dry_run:
        print('Dry run: no deletes performed. Re-run with dry_run=False to delete.')
        return study_ids

    deleted = []
    for study_id in study_ids:
        print('Deleting study:', study_id)
        try:
            resp = orthanc_delete(f'/studies/{study_id}')
            if resp['status'] == 200:
                deleted.append(study_id)
            else:
                print('Failed to delete study', study_id, resp['status'], resp.get('text'))
        except Exception as e:
            print('Delete failed for', study_id, 'with error:', type(e).__name__, e)
        time.sleep(DELETE_PAUSE_SECONDS)
    return deleted


# Dry run example for one patient
# to_delete = delete_studies_for_patient_id('40152149', dry_run=True)
# print('Study IDs for patient to delete:', to_delete)


In [ ]:
# Dry run: list studies that would be deleted
studies_to_delete = delete_studies_not_in_allowed_patient_ids(allowed_patient_ids, dry_run=False)
print('Candidate studies to delete (dry run):', len(studies_to_delete))
print(studies_to_delete)

# Uncomment the next line to actually delete them after verifying the dry run
# delete_studies_not_in_allowed_patient_ids(allowed_patient_ids, dry_run=False)
